#FDIC Community Banks Python API Requests 

In [ ]:
##Pull data from FDIC API for all active banks in California and save to CSV

import requests
import pandas as pd

url = "https://banks.data.fdic.gov/api/institutions"

params = {
    "filters": "STALP:CA AND ACTIVE:1",
    "fields": "CERT,NAME,CITY,STALP,ESTYMD,BKCLASS",
    "limit": 10000,
    "format": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()
institutions = pd.json_normalize([record["data"] for record in data["data"]])

print(f"Pulled {len(institutions)} California institutions")
print(institutions.head())

institutions.to_csv("ca_institutions.csv", index=False)


In [ ]:
ca_institutions = pd.read_csv("ca_institutions.csv")

for col in ca_institutions.select_dtypes(include="object").columns:
   ca_institutions[col] = ca_institutions[col].astype(str).str.replace('"', "'", regex=False)


ca_institutions.to_csv("ca_institutions_reformatted.csv", index=False)

In [ ]:
## Pull financial data for all active banks in California and save to CSV

import time

institutions = pd.read_csv("ca_institutions.csv")
certs = institutions["CERT"].astype(str).tolist()

url = "https://banks.data.fdic.gov/api/financials"
fields = "CERT,REPDTE,ASSET,DEP,NETINC,ROA,ROE,EEFFR,LNRE,LNCI,LNCON,NUMEMP"

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

all_financials = []

for batch in chunk_list(certs, 50):
    cert_filter = " OR ".join([f"CERT:{c}" for c in batch])
    filters = f"({cert_filter}) AND REPDTE:[20220101 TO 20251231]"

    params = {
        "filters": filters,
        "fields": fields,
        "limit": 10000,
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    records = [r["data"] for r in data["data"]]
    all_financials.extend(records)

    time.sleep(0.2)

financials = pd.json_normalize(all_financials)

print(f"Pulled {len(financials)} bank-quarter records")
print(financials.head())

financials.to_csv("ca_financials_2022_2025.csv", index=False)


In [ ]:
ca_financials = pd.read_csv("ca_financials_2022_2025.csv")

for col in ca_financials.select_dtypes(include="object").columns:
   ca_financials[col] = ca_financials[col].astype(str).str.replace('"', "'", regex=False)


ca_financials.to_csv("ca_financials_reformatted.csv", index=False)

In [ ]:
### Pull California GDP data from BEA API and save to CSV

import os

api_key = api_key = os.environ.get("BEA_API_KEY")
url = "https://apps.bea.gov/api/data"

params = {
    "UserID": api_key,
    "method": "GetData",
    "datasetname": "Regional",
    "TableName": "SQGDP1",
    "LineCode": 1,
    "GeoFips": "06000",
    "Year": "2022,2023,2024,2025",
    "ResultFormat": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

records = data["BEAAPI"]["Results"]["Data"]
state_gdp = pd.DataFrame(records)

print(f"Pulled {len(state_gdp)} rows")
print(state_gdp.head())

state_gdp.to_csv("ca_state_gdp_2022_2025.csv", index=False)


In [ ]:
ca_state_gdp = pd.read_csv("ca_state_gdp_2022_2025.csv")

ca_state_gdp["STALP"] = "CA"

for col in ca_state_gdp.select_dtypes(include="object").columns:
   ca_state_gdp[col] = ca_state_gdp[col].astype(str).str.replace('"', "'", regex=False)


ca_state_gdp.to_csv("ca_state_gdp_reformatted.csv", index=False)